# Week 1 | Employee Attrition Evidence Lab

**Course:** 24ADI204 Data Science and Visualization · **Team No. 3**

This is a fresh start using the supplied **Employee Attrition Uncleaned Dataset**.
The previous IBM dataset, lab logs and presentation are legacy materials.
No historical model accuracy is evidence for this project.

**Research question:** Which workforce patterns are associated with the `Left` label,
and which conclusions remain credible after reasonable cleaning and subgroup checks?

**Week 1 outcome:** define the scope, document the source, establish reproducible loading,
and introduce Pandas/NumPy. Run every notebook from top to bottom in the Python environment
described in the project README. Each notebook loads its own data and can run independently.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Works when opened from the notebook folder, DSV_PROJECT, or repository root.
ROOT = next((p if (p/'Source_Code'/'attrition_lab.py').exists() else p/'DSV_PROJECT'
             for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'Source_Code'/'attrition_lab.py').exists()
             or (p/'DSV_PROJECT'/'Source_Code'/'attrition_lab.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the DSV_PROJECT workspace.')
sys.path.insert(0, str(ROOT/'Source_Code'))
from attrition_lab import (load_raw, clean_data, quality_report, outlier_report,
                          rate_table, compare, evidence_tables, scenarios,
                          build_artifacts, NUMERIC, MISSING, ORDERS, CONTRASTS)
raw = load_raw()
sns.set_theme(style='whitegrid', palette=['#176B87','#DA7652','#71AFA0','#A18CB8'])
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.titlesize':13, 'font.size':10})
FIGURES = ROOT/'Reports'/'Figures'
FIGURES.mkdir(parents=True, exist_ok=True)
def finish(name):
    plt.tight_layout()
    plt.savefig(FIGURES/f'{name}.png', dpi=145, bbox_inches='tight')
    plt.show()
    plt.close()
pd.set_option('display.max_columns', 12)

## Source and boundaries

[Nikhil Bhosle — Employee Attrition Uncleaned Dataset on Kaggle](https://www.kaggle.com/datasets/nikhilbhosle/employee-attrition-uncleaned-dataset)

The local CSV was supplied by the team; the team confirmed faculty approval. We did not
add missing values, duplicate records or artificial labels. Kaggle's API lists its license
as “Other (specified in description)”; we do not assign it a different license.
The source's sampling frame, collection dates, currency, distance units, original employer
and real-versus-synthetic origin are not verified. The title alone does not establish those facts.

There is no time window or headcount history. Throughout this book, **left share** means
the fraction of supplied records labelled `Left`, not an annual attrition rate or a probability
that a current employee will leave. Employee ID is only an audit key.

In [2]:
import hashlib
from attrition_lab import RAW
display(pd.DataFrame([{'rows':len(raw), 'columns':raw.shape[1],
                       'file_sha256':hashlib.sha256(RAW.read_bytes()).hexdigest()}]))
display(raw.head())

,rows,columns,file_sha256
0,74610,24,827b58b28f2f3bcae9e3d04208f49b32931d8fe5db431b...


,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,...,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,Male,19,Education,5390,...,No,No,No,Excellent,Medium,Stayed
1,64756,59,Female,4,Media,5534,...,No,No,No,Fair,Low,Stayed
2,30257,24,Female,10,Healthcare,8159,...,No,No,No,Poor,Low,Stayed
3,65791,36,Female,7,Education,3989,...,Yes,No,No,Good,Medium,Stayed
4,65026,56,Male,41,Education,4821,...,No,No,No,Fair,Medium,Stayed


## First Pandas and NumPy operations

Inspect the first rows, select columns, calculate counts and compare mean versus median.
At this stage these are **raw** summaries, including duplicates and unhandled anomalies.

In [3]:
display(raw[['Age','Monthly Income','Attrition']].head(8))
income = raw['Monthly Income'].to_numpy()
display(pd.Series({'raw_mean_income':np.mean(income), 'raw_median_income':np.median(income),
                   'raw_left_count':raw['Attrition'].eq('Left').sum()}))
display(raw['Attrition'].value_counts().rename('records').to_frame())

,Age,Monthly Income,Attrition
0,31,5390,Stayed
1,59,5534,Stayed
2,24,8159,Stayed
3,36,3989,Stayed
4,56,4821,Stayed
5,38,9977,Left
6,47,3681,Left
7,48,11223,Stayed


raw_mean_income       7344.931417
raw_median_income     7348.500000
raw_left_count       35419.000000
dtype: float64

,records
Attrition,
Stayed,39191
Left,35419


## A demonstrable distinguishing feature

Commercial services already offer turnover predictions and driver analysis:
[Workday employee retention](https://www.workday.com/en-us/products/employee-voice/employee-retention.html)
and [Visier talent retention](https://www.visier.com/products/talent-retention/).
Prediction, feature importance and an HR dashboard alone are therefore not our novelty claim.

Our proposed contribution is a reproducible **evidence check attached to each EDA claim**:

1. Show group counts, observed differences and uncertainty.
2. Compare five explicit data-cleaning scenarios.
3. Check directions within job roles and standardize by role and level.
4. Label weak or unstable claims `Needs caution`, with the reason visible.

This is a distinctive project implementation, not proof that no commercial service or
research paper has ever used these methods. Vendor pages describe selected public features;
absence of a feature there does not establish its absence from the product.

## Week 1–4 deliverable map

| Week | Notebook | Evidence produced |
|---|---|---|
| 1 | Project and data source | Research question, source, basic operations |
| 2 | Know your data | Types, missingness, duplicate and consistency audit |
| 3 | Cleaning sprint | Median imputation, alternative strategies, IQR and Z-score flags |
| 4 | EDA book | Distributions, comparisons, interpretations and evidence scorecard |

Scaling/encoding, PCA, predictive models and a production dashboard remain future work.
The README includes the VS Code setup and GitHub workflow.